In [1]:
from pathlib import Path
import json

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import pandas as pd
from tqdm import tqdm
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

c:\Users\Moritz\miniconda3\envs\phishing-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RESULTS_DIR = BASE_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
ERROR_DIR = RESULTS_DIR / "errors"

In [3]:
ERROR_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
class URLDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.urls = df["url"].astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = self.urls[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            url,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }


def bert_predict(df, tokenizer, model, device, batch_size=64, max_len=64):
    ds = URLDataset(df, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size)

    model.eval()
    preds, probs, labels = [], [], []

    with torch.no_grad():
        for batch in tqdm(dl, total=len(dl), desc="BERT eval"):
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            out = model(input_ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:, 1]

            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            probs.extend(prob.cpu().tolist())
            labels.extend(label.cpu().tolist())

    return preds, probs, labels

In [5]:

test_df = pd.read_csv(PROCESSED_DIR / "urls_test.csv")
len(test_df), test_df.head()


(110170,
                                                  url  label  source
 0                      bestindiansites.com/hardware/      0  github
 1        www.attivita-antroposofiche-roma.org/kw9qzt      1  github
 2                 muti.site11.com/facebook/login.php      1  github
 3                                 sricar.com/office/      1  github
 4  virtualtourist.com/travel/North_America/United...      0  github)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_DIR / "models" / "bert")
model = DistilBertForSequenceClassification.from_pretrained(BASE_DIR / "models" / "bert").to(device)

bert_preds, bert_probs, bert_labels = bert_predict(test_df, tokenizer, model, device)

bert_df = test_df.copy()
bert_df["bert_pred"] = bert_preds
bert_df["bert_proba"] = bert_probs

def classify_error(row):
    if row["label"] == 1 and row["bert_pred"] == 1:
        return "TP"
    if row["label"] == 0 and row["bert_pred"] == 0:
        return "TN"
    if row["label"] == 0 and row["bert_pred"] == 1:
        return "FP"
    if row["label"] == 1 and row["bert_pred"] == 0:
        return "FN"
    return "OTHER"

bert_df["error_type"] = bert_df.apply(classify_error, axis=1)
bert_df["error_type"].value_counts()

BERT eval:  18%|█▊        | 302/1722 [10:36<44:02,  1.86s/it] 

In [ ]:
bert_df.to_csv(ERROR_DIR / "bert_errors_all.csv", index=False)
bert_df[bert_df["error_type"] == "FP"].to_csv(ERROR_DIR / "bert_false_positives.csv", index=False)
bert_df[bert_df["error_type"] == "FN"].to_csv(ERROR_DIR / "bert_false_negatives.csv", index=False)

ERROR_DIR, [p.name for p in ERROR_DIR.glob("bert_*.csv")]

In [ ]:
train_feat = pd.read_csv(PROCESSED_DIR / "urls_train_features.csv")
test_feat = pd.read_csv(PROCESSED_DIR / "urls_test_features.csv")

N_TRAIN = 20000
N_TEST = 50000

if len(train_feat) > N_TRAIN:
    train_feat = train_feat.sample(n=N_TRAIN, random_state=42).reset_index(drop=True)
if len(test_feat) > N_TEST:
    test_feat = test_feat.sample(n=N_TEST, random_state=42).reset_index(drop=True)

len(train_feat), len(test_feat)

In [ ]:
class URLInferDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.urls = df["url"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = self.urls[idx]
        enc = self.tokenizer(
            url,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }


def get_bert_features(df, tokenizer, model, device, batch_size=64, max_len=64):
    ds = URLInferDataset(df, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size)

    model.eval()
    all_logits, all_probs = [], []

    with torch.no_grad():
        for batch in tqdm(dl, total=len(dl), desc="BERT feats"):
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            out = model(input_ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:, 1]

            all_logits.extend(logits[:, 1].cpu().tolist())
            all_probs.extend(prob.cpu().tolist())

    return pd.DataFrame({"bert_logit": all_logits, "bert_proba": all_probs})

In [ ]:

bert_train_feats = get_bert_features(train_feat, tokenizer, model, device)
bert_test_feats = get_bert_features(test_feat, tokenizer, model, device)

train_hybrid = pd.concat([train_feat.reset_index(drop=True), bert_train_feats], axis=1)
test_hybrid = pd.concat([test_feat.reset_index(drop=True), bert_test_feats], axis=1)

train_hybrid.head()

In [ ]:
drop_cols = ["url", "label"]
feature_cols = [c for c in train_hybrid.columns if c not in drop_cols]

X_train = train_hybrid[feature_cols]
y_train = train_hybrid["label"]

X_test = test_hybrid[feature_cols]
y_test = test_hybrid["label"]

lgb = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42,
)
lgb.fit(X_train, y_train)

hyb_pred = lgb.predict(X_test)
hyb_proba = lgb.predict_proba(X_test)[:, 1]

metrics_hyb = {
    "accuracy": accuracy_score(y_test, hyb_pred),
    "f1": f1_score(y_test, hyb_pred),
    "roc_auc": roc_auc_score(y_test, hyb_proba),
    "pr_auc": average_precision_score(y_test, hyb_proba),
}
metrics_hyb

In [ ]:

hyb_df = test_hybrid.copy()
hyb_df["hyb_pred"] = hyb_pred
hyb_df["hyb_proba"] = hyb_proba

def classify_error_h(row):
    if row["label"] == 1 and row["hyb_pred"] == 1:
        return "TP"
    if row["label"] == 0 and row["hyb_pred"] == 0:
        return "TN"
    if row["label"] == 0 and row["hyb_pred"] == 1:
        return "FP"
    if row["label"] == 1 and row["hyb_pred"] == 0:
        return "FN"
    return "OTHER"

hyb_df["error_type"] = hyb_df.apply(classify_error_h, axis=1)
hyb_df["error_type"].value_counts()

In [ ]:
hyb_df.to_csv(ERROR_DIR / "hybrid_errors_all.csv", index=False)
hyb_df[hyb_df["error_type"] == "FP"].to_csv(ERROR_DIR / "hybrid_false_positives.csv", index=False)
hyb_df[hyb_df["error_type"] == "FN"].to_csv(ERROR_DIR / "hybrid_false_negatives.csv", index=False)

[p.name for p in ERROR_DIR.glob("hybrid_*.csv")]